# IoT/IIoT Intrusion Detection — Edge-IIoTset
### Random Forest vs XGBoost (Colab version)

This notebook is the Colab-friendly companion to `intrusion_detection.py` in this
repository. Both files do the same thing — use whichever fits your workflow
(see README.md, and Lab Part A3 on `.py` vs `.ipynb`).


In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Libraries loaded.")

## utils.py equivalent
(In the `.py` version this is imported from `utils.py`. Defined inline here so the
notebook runs standalone in Colab.)

In [ ]:
def check_overfitting(train_acc, test_acc, threshold=0.05):
    for name, value in (('train_acc', train_acc), ('test_acc', test_acc)):
        if not (0.0 <= value <= 1.0):
            raise ValueError(f"{name} must be between 0.0 and 1.0, got {value}")
    gap = train_acc - test_acc
    return gap, gap > threshold

## Step 1 — Load Dataset
Download `ML-EdgeIIoT-dataset.csv` (Kaggle or IEEE DataPort — see README.md) and
upload it below, or mount Google Drive if you saved it there.

In [ ]:
from google.colab import files
print("Upload ML-EdgeIIoT-dataset.csv (or skip if already in /content/dataset/):")
uploaded = files.upload()

In [ ]:
DATA_PATH = "ML-EdgeIIoT-dataset.csv"
matches = glob.glob("**/*ML-EdgeIIoT*.csv", recursive=True)
if matches:
    DATA_PATH = matches[0]

print("Using:", DATA_PATH)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape:", df.shape)
df.head()

## Step 2 — Preprocessing

In [ ]:
drop_candidates = ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.src.proto_ipv4',
                    'arp.dst.proto_ipv4', 'http.file_data', 'http.request.full_uri',
                    'icmp.transmit_timestamp', 'http.request.uri.query', 'tcp.options',
                    'tcp.payload', 'tcp.srcport', 'tcp.dstport', 'udp.port', 'mqtt.msg']
existing_drop_cols = [c for c in drop_candidates if c in df.columns]
df = df.drop(columns=existing_drop_cols)

TARGET = 'Attack_type'
OTHER_LABEL = 'Attack_label' if TARGET == 'Attack_type' else 'Attack_type'
if OTHER_LABEL in df.columns:
    df = df.drop(columns=[OTHER_LABEL])

df = df.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates()
print("Rows after cleaning:", len(df))

X = df.drop(columns=[TARGET])
y = df[TARGET]
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

le_target = LabelEncoder()
y_enc = le_target.fit_transform(y)
class_names = le_target.classes_
print("Classes:", list(class_names))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=RANDOM_STATE, stratify=y_enc
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 3 — Train Random Forest & XGBoost

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=150, max_depth=14, min_samples_leaf=3,
    random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)
print("Random Forest trained.")

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss',
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(X_train_scaled, y_train)
print("XGBoost trained.")

## Step 4 — Evaluate

In [ ]:
def evaluate_model(model, name, X_tr, y_tr, X_te, y_te, classes):
    y_train_pred = model.predict(X_tr)
    y_test_pred = model.predict(X_te)
    train_acc = accuracy_score(y_tr, y_train_pred)
    test_acc = accuracy_score(y_te, y_test_pred)
    gap, is_overfitting = check_overfitting(train_acc, test_acc)

    precision = precision_score(y_te, y_test_pred, average='weighted', zero_division=0)
    recall = recall_score(y_te, y_test_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_te, y_test_pred, average='weighted', zero_division=0)
    y_proba = model.predict_proba(X_te)
    y_test_bin = label_binarize(y_te, classes=np.arange(len(classes)))
    roc_auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')

    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Gap: {gap:.4f} {'(OK)' if not is_overfitting else '(WARNING: overfitting)'}")
    print(f"Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}  ROC-AUC: {roc_auc:.4f}")
    return {'name': name, 'train_acc': train_acc, 'test_acc': test_acc, 'gap': gap,
            'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc,
            'y_pred': y_test_pred}

results_rf = evaluate_model(rf_model, "Random Forest", X_train_scaled, y_train, X_test_scaled, y_test, class_names)
results_xgb = evaluate_model(xgb_model, "XGBoost", X_train_scaled, y_train, X_test_scaled, y_test, class_names)

In [ ]:
comparison = pd.DataFrame([
    {'model': r['name'], 'accuracy': r['test_acc'], 'precision': r['precision'],
     'recall': r['recall'], 'f1_score': r['f1'], 'roc_auc': r['roc_auc'],
     'overfitting_gap': r['gap']}
    for r in [results_rf, results_xgb]
])
comparison.to_csv(os.path.join(OUTPUT_DIR, 'model_results.csv'), index=False)
print(comparison.round(4))

In [ ]:
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
x = np.arange(len(metrics_to_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, comparison.loc[0, metrics_to_plot], width, label='Random Forest')
ax.bar(x + width/2, comparison.loc[1, metrics_to_plot], width, label='XGBoost')
ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot); ax.set_ylim(0, 1)
ax.set_title('Random Forest vs XGBoost — Performance Comparison')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

## Step 5 — Confusion Matrices & Classification Reports

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, res, title in zip(axes, [results_rf, results_xgb], ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names, cbar=False)
    ax.set_title(f'{title} — Confusion Matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

In [ ]:
print("RANDOM FOREST\n", classification_report(y_test, results_rf['y_pred'], target_names=class_names, zero_division=0))
print("XGBOOST\n", classification_report(y_test, results_xgb['y_pred'], target_names=class_names, zero_division=0))

## Step 6 — Export Best Model as .pkl

In [ ]:
best_model, best_name = (rf_model, "RandomForest") if results_rf['f1'] >= results_xgb['f1'] else (xgb_model, "XGBoost")
print(f"Best model: {best_name} (Test F1 = {max(results_rf['f1'], results_xgb['f1']):.4f})")

deployment_bundle = {
    'model': best_model, 'model_name': best_name, 'scaler': scaler,
    'label_encoder': le_target, 'feature_columns': list(X.columns),
    'class_names': list(class_names)
}
joblib.dump(deployment_bundle, os.path.join(OUTPUT_DIR, 'iot_intrusion_detector.pkl'))
print(f"Saved: {OUTPUT_DIR}/iot_intrusion_detector.pkl")

from google.colab import files
files.download(os.path.join(OUTPUT_DIR, 'iot_intrusion_detector.pkl'))